# Point-Source Spectral Analysis with Gammapy  
### Example: The Crab Nebula

In this tutorial, we perform a 1D spectral analysis of the **Crab Nebula** using [Gammapy](https://gammapy.org/).  

---

## Objectives
- Learn how to access and select Crab observations with `DataStore`.
- Build spectral datasets with Gammapy makers.
- Fit a simple power-law model to the Crab spectrum.
- Compute flux points and compare with the fitted model.


In [ ]:
import logging
import numpy as np
from copy import deepcopy
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, Angle
from regions import CircleSkyRegion
from scipy.stats import chi2


from gammapy.data import DataStore
from gammapy.maps import MapAxis, RegionGeom,WcsGeom
from gammapy.makers import SpectrumDatasetMaker, ReflectedRegionsBackgroundMaker, SafeMaskMaker
from gammapy.datasets import Datasets, SpectrumDataset
from gammapy.modeling.models import PowerLawSpectralModel, SkyModel, ExpCutoffPowerLawSpectralModel, LogParabolaSpectralModel, Models
from gammapy.modeling import Fit
from gammapy.estimators import FluxPointsEstimator
from gammapy.visualization import plot_spectrum_datasets_off_regions
from gammapy.stats.utils import ts_to_sigma


# Logging setup
logging.basicConfig()
log = logging.getLogger("gammapy-tutorial")
log.setLevel(logging.INFO)


---

## Accessing Crab Observations

We use the **H.E.S.S. DL3 public dataset** that comes with Gammapy.  
The Crab is included with 4 observations.


In [ ]:
# Load the demo H.E.S.S. data
datastore = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1")

# Get Crab coordinates directly from its name
crab_position = SkyCoord.from_name("Crab")

print(crab_position)

# Select all observations within 2° of the Crab
obs_table_filtered = datastore.obs_table.select_sky_circle(
    center=crab_position, radius=2 * u.deg
)

# Extract IDs
obs_ids = obs_table_filtered["OBS_ID"]

# Retrieve observations
observations = datastore.get_observations(obs_ids)

print(f"Number of selected observations: {len(observations)}")

**Note** Here we selected all observations whose pointing direction is within 2° of the Crab.
This is not the ON region size -- it’s just a way to make sure we load all runs where the Crab was observed, even in wobble mode. The choice of 2° is motivated by the fact that H.E.S.S. has a field of view of about 5°, so the Crab will still be well within the camera. The actual source extraction region (ON region) will be much smaller, typically 0.1°, which is comparable to the instrument’s PSF.

In [ ]:
#Let's check the observations
obs_table_filtered

If you wish to apply more complex filtering options, you can use the `~gammapy.data.ObservationTable.select_observations()` method instead. This provides the freedom of selecting observations based on a sky circle, time period or parameter (e.g. Zenith angle) range.

## Choosing a representative observation & peeking at data/IRFs

Before building datasets, it’s useful to **inspect at least one observation** in detail:
- `events.peek()` → quick look at event distributions (energy, offset, etc.)
- `aeff.peek()` → effective area (IRF)
- `edisp.peek()` → energy dispersion (IRF)
- `psf.peek()` → point-spread function (IRF)
- `bkg.peek()` → background model (if provided in DL3)

We’ll pick the observation whose pointing is **closest to the Crab** (smallest offset) as a representative run to inspect.


In [ ]:
#First obs in our list
obs_1 = observations[0]

#Check
print(type(obs_1), obs_1.obs_id)

In [ ]:
# Events: energy / offset 
obs_1.events.peek()

In [ ]:
# Effective area (AEFF)
obs_1.aeff.peek()

In [ ]:
# Energy dispersion (EDISP)
obs_1.edisp.peek()


In [ ]:
# PSF
obs_1.psf.peek()


In [ ]:
# Background template (se existir no DL3)
#if getattr(obs_1, "bkg", None) is not None:
 #   obs_1.bkg.peek()
#else:
 #   print("This DL3 dataset does not provide a background template as `obs_1.bkg`.")


## Preparing reduced datasets geometry

Now we define the [reconstructed](https://docs.gammapy.org/1.1/user-guide/references.html#term-Reco-Energy) and [true](https://docs.gammapy.org/1.1/user-guide/references.html#term-True-Energy) energy axes: 

In [ ]:
# Reconstructed energy: 0.1–40 TeV, 10 bins/decade
energy_axis = MapAxis.from_energy_bounds(
    0.1, 40, nbin=10, per_decade=True, unit="TeV", name="energy"
)

# True energy: extend a bit beyond reco (IRF tails), 0.05–100 TeV, 20 bins/decade
energy_axis_true = MapAxis.from_energy_bounds(
    0.05, 100, nbin=20, per_decade=True, unit="TeV", name="energy_true"
)
energy_axis, energy_axis_true

Define the ON region and create the analysis geometry

We use a **0.11°** circular ON region (typical for point-source 1D analyses with H.E.S.S./CTA).  
Then we build a `RegionGeom` combining the region and the reconstructed-energy axis.


In [ ]:
# ON region centered on the Crab
on_region_radius = Angle("0.11 deg")
on_region = CircleSkyRegion(center=crab_position, radius=on_region_radius)

# Geometry for spectral extraction
geom = RegionGeom.create(region=on_region, axes=[energy_axis])
geom

## Create an empty SpectrumDataset

This template will be filled per observation by the makers in the next step.


In [ ]:
from gammapy.datasets import SpectrumDataset

dataset_empty = SpectrumDataset.create(
    geom=geom,
    energy_axis_true=energy_axis_true
)
print(dataset_empty)


## Optional: create an exclusion mask (for reflected OFF placement)

For crowded fields you may want to exclude known gamma-ray sources (or bright stars) from the OFF placement.  
Here we illustrate how to build an exclusion mask, even though for the Crab field it’s usually not required.


In [ ]:
# Build a WCS geometry that covers the camera FoV around the Crab
# (5° wide, TAN projection is fine for small fields)
skydir = crab_position.icrs
exgeom = WcsGeom.create(
    skydir=skydir, width=6.5 * u.deg, binsz=0.02, proj="TAN", frame="icrs"
)

# Example exclusion region:
# a circle 0.5 deg to the North of the Crab, radius 0.3 deg.
# (Replace with real neighbor source positions if needed.)
excl_center = SkyCoord(crab_position.ra, crab_position.dec + 2 * u.deg, frame="icrs")
exclusion_region = CircleSkyRegion(center=excl_center, radius=0.3 * u.deg)

# Build a boolean mask: region_mask is True *inside* the region;
# for an exclusion mask we invert it so that False marks the excluded area.
region_mask = exgeom.region_mask([exclusion_region])
exclusion_mask = ~region_mask

ax = exclusion_mask.plot(add_cbar=False)

# WCS to ICRS
ax.scatter(
    crab_position.ra.deg,
    crab_position.dec.deg,
    s=80, marker="+",
    transform=ax.get_transform('icrs')
)

## Data reduction

### Create the maker classes to be used
We first initialize the `Maker` objects that will take care of the data reduction.

In [ ]:
dataset_maker = SpectrumDatasetMaker(
    containment_correction=True, selection=["counts", "exposure", "edisp"]
)

The `~gammapy.makers.ReflectedRegionsBackgroundMaker` appends a background estimate (based on the [reflected regions](https://docs.gammapy.org/1.0.1/user-guide/makers/reflected.html?highlight=reflected) method) to an input `SpectrumDataset`, converting it into a `SpectrumDatasetOnOff`.

In [ ]:
#bkg_maker = ReflectedRegionsBackgroundMaker(exclusion_mask=exclusion_mask) 
bkg_maker = ReflectedRegionsBackgroundMaker() 

Safe energy range:
"aeff-max" removes bins where the effective area is too low (default threshold 10%)
You can later add "edisp-bias" if you want to limit energy bias (e.g., bias_percent=10)

In [ ]:
safe_mask_maker = SafeMaskMaker(
    methods=["aeff-max"],   # optionally add "edisp-bias"
    aeff_percent=10         # keep energies with >=10% of max Aeff
    # bias_percent=10,      # uncomment if using methods=["aeff-max","edisp-bias"]
)


### Build per-observation spectral datasets (1D, ON/OFF)

We now convert each selected observation into a `SpectrumDataset` on our region/energy geometry:

1) **`SpectrumDatasetMaker`** fills counts and IRFs (exposure + energy dispersion).  
2) **`ReflectedRegionsBackgroundMaker`** places **reflected OFF regions** to estimate the background.  
3) **`SafeMaskMaker`** applies **safe energy cuts** (e.g., remove bins with very low effective area).


In [ ]:
datasets = Datasets()

for obs in observations:
    # Empty template on our geometry for each observation
    dataset = SpectrumDataset.create(
        geom=geom,
        energy_axis_true=energy_axis_true,
        name=f"obs-{int(obs.obs_id)}"
    )
    
    # 1) counts + exposure + edisp
    dataset = dataset_maker.run(dataset, obs)
    
    # 2) reflected OFF background (mask already attached in bkg_maker if you set one)
    dataset = bkg_maker.run(dataset, obs)
    
    # 3) safe energy cuts
    dataset = safe_mask_maker.run(dataset, obs)

    datasets.append(dataset)

print(f"Built {len(datasets)} spectral datasets")
[d.name for d in datasets]


### Quick sanity checks
Let’s inspect one dataset and the OFF-region placement to ensure the reduction looks sensible.


In [ ]:
# Inspect the first dataset
d0 = datasets[0]
print(d0)


### Visualizing ON and OFF regions

After applying the `ReflectedRegionsBackgroundMaker`, each `SpectrumDatasetOnOff` contains, in addition to the **ON region** (defined around the Crab position), a set of **reflected OFF regions**. These OFF regions are placed automatically by the algorithm: they have the **same radius** as the ON region but are rotated around the **pointing position** of each observation, sampling the background in regions expected to be free of gamma-ray emission.

In the plot below, each color corresponds to a different observation (`obs_id`):
- The **black circle** marks the ON region centered on the Crab.
- The **colored circles** show the reflected OFF regions for each observation.


This visualization is an important diagnostic step:
- If any OFF regions overlap with known astrophysical emission, apply an **exclusion mask** to remove them.
- Otherwise, the background estimation can proceed with the default configuration (no mask).


In [ ]:
ax = exclusion_mask.plot();
plot_spectrum_datasets_off_regions(ax=ax, datasets=datasets);
on_region.to_pixel(ax.wcs).plot(ax=ax, color="black");

## Data fitting and hypothesis testing

We compare three hypotheses using the (profile) likelihood ratio:

- **H0**: background-only (no source)
- **H1**: background + source described by a **PowerLaw**
- **H2**: background + source described by a **PowerLaw with exponential cutoff**

Let `TS = 2(ln L_alt − ln L_null)`.  
Because Gammapy uses `stat = −2 ln L`, this becomes **`TS = stat_null − stat_alt`**.

- **Detection test**: `TS_det = stat(H0) − stat(H1)`  
  
- **Cutoff test**: `TS_cut = stat(H1) − stat(H2)`  

We’ll fit H1 and H2, and evaluate the statistic for H0 by setting **no source model**.


Before fitting we stack the `Datasets` into a single `SpectrumDatasetOnOff`:

In [ ]:
stacked = datasets.stack_reduce(name="crab-stacked")

### **H0**

The value of the quantity $-2\ln\mathcal(L)$ for the background-only model (null hypothesis) can be simply computed as

In [ ]:
Wstat_0 = stacked.stat_sum()
print(Wstat_0)

We can inspect the model residuals for the H0 hypothesis:

In [ ]:
stacked.plot_residuals_spectral()

As expected, the residuals show a clear positive feature indicating that a source is missing in the model. 

### **H1**
We now add a source defined by a power law spectrum 

In [ ]:
lp = LogParabolaSpectralModel(
    amplitude=1e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference=1 * u.TeV,
    alpha=2.4,  # slope at E_ref
    beta=0.1,   # curvature
)
model_lp = SkyModel(spectral_model=lp, name="crab_lp")

model_lp.parameters["alpha"].frozen = True

In [ ]:
model_lp.parameters.to_table()

Now we assign the model to our stacked dataset

In [ ]:
stacked.models = [model_lp]

stacked.models

And run the fit

In [ ]:
fit1 = Fit(optimize_opts={"print_level": 1})
result1 = fit1.run(datasets=[stacked])

print(result1.success)
print(result1.minuit)

In [ ]:
result1.models.to_parameters_table()

In [ ]:
Wstat_1 = result1.total_stat
print(Wstat_1)
delta_ts_1 = Wstat_0-Wstat_1
df = len(result1.models.parameters.free_parameters.names)
sigma = ts_to_sigma(delta_ts_1, df=df)
print(f"The delta_ts  of H1 vs H0: {delta_ts_1:.3f}, that gives a p-value of {chi2.sf(delta_ts_1, df)}")
print(f"Converting this to a significance gives: {sigma:.3f} \u03C3")

In [ ]:
ax_spectrum, ax_residuals = stacked.plot_fit()

We can compute flux points for the H1 model assumption.

In [ ]:
energy_edges = np.logspace(-1, 1.5, 12) * u.TeV  # 0.1–16 TeV
fpe = FluxPointsEstimator(energy_edges=energy_edges, source="crab_lp", selection_optional=["ul"])
flux_points = fpe.run(datasets=[stacked])

In [ ]:
emin, emax = energy_edges[0], energy_edges[-1]
ax = flux_points.plot(sed_type="dnde", marker="o", color="black", label="Flux points (stacked)")

stacked.models["crab_lp"].spectral_model.plot(
    energy_bounds=(emin, emax), ax=ax, color="tab:orange", label="LogParabola (fit)"
)
stacked.models["crab_lp"].spectral_model.plot_error(
    energy_bounds=(emin, emax), ax=ax, color="tab:orange", alpha=0.25
)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$E$ [TeV]"); ax.set_ylabel(r"$\mathrm{d}N/\mathrm{d}E$ [cm$^{-2}$ s$^{-1}$ TeV$^{-1}$]")
ax.legend(); plt.show()

### **H2**
We now add a source defined by an ECPL 

In [ ]:
# Restore and define H2
stacked.models = []  # start clean

ecpl = ExpCutoffPowerLawSpectralModel(
    index=2.0,
    amplitude=3e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference=1 * u.TeV,
    lambda_=1.0 / u.TeV,  # cutoff scale; E_cut = 1/lambda_
    # (Alternatively: use 'cutoff' parameter if you prefer ExpCutoffPowerLawSpectralModel with cutoff)
)
model_ecpl = SkyModel(spectral_model=ecpl, name="crab_ecpl")

In [ ]:
model_ecpl.parameters.to_table()

In [ ]:
stacked.models = [model_ecpl]

stacked.models

In [ ]:
fit2 = Fit(optimize_opts={"print_level": 1})
result2 = fit2.run(datasets=[stacked])

print(result2.success)
print(result2.minuit)

In [ ]:
result2.models.to_parameters_table()

In [ ]:
Wstat_2 = result2.total_stat
print(Wstat_2)
delta_ts_2 = Wstat_0-Wstat_2
df = len(result2.models.parameters.free_parameters.names)
sigma = ts_to_sigma(delta_ts_2, df=df)
print(f"The delta_ts  of H1 vs H0: {delta_ts_2:.3f}, that gives a p-value of {chi2.sf(delta_ts_2, df)}")
print(f"Converting this to a significance gives: {sigma:.3f} \u03C3")

In [ ]:
energy_edges = np.logspace(-1, 1.5, 12) * u.TeV  # 0.1–16 TeV
fpe = FluxPointsEstimator(energy_edges=energy_edges, source="crab_ecpl", selection_optional=["ul"])
flux_points = fpe.run(datasets=[stacked])

In [ ]:
emin, emax = energy_edges[0], energy_edges[-1]
ax = flux_points.plot(sed_type="dnde", marker="o", color="black", label="Flux points (stacked)")

stacked.models["crab_ecpl"].spectral_model.plot(
    energy_bounds=(emin, emax), ax=ax, color="tab:orange", label="ECPL (fit)"
)
stacked.models["crab_ecpl"].spectral_model.plot_error(
    energy_bounds=(emin, emax), ax=ax, color="tab:orange", alpha=0.25
)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$E$ [TeV]"); ax.set_ylabel(r"$\mathrm{d}N/\mathrm{d}E$ [cm$^{-2}$ s$^{-1}$ TeV$^{-1}$]")
ax.legend(); plt.show()

In [ ]:
Wstat_2 = result2.total_stat
delta_ts = Wstat_1-Wstat_2
df = len(result2.models.parameters.free_parameters.names)-len(result1.models.parameters.free_parameters.names)
print(df)
sigma = ts_to_sigma(delta_ts, df=df)
print(f"The delta_ts  of H2 vs H1: {delta_ts:.3f}, that gives a p-value of {chi2.sf(delta_ts, df)}")
print(f"Converting this to a significance gives: {sigma:.3f} \u03C3")

We have no significant preference between the LP model and ECPL for this dataset

## Exercises:

- Try other models, eg: log-parabola, broken power law, etc. See the model gallery for a list of available models: https://docs.gammapy.org/1.1/user-guide/model-gallery/index.html
- Select and analyze observations of another source (for example, PSK 2155-304 during its steady state). 